# 14 · Prefix caching: hashes, block chains and hit rate
Level L3 · Part 2 · core · ~25 min · builds on 11, 12

## Goal
By the end of this lesson you can explain how a block's hash chains over the prefix before it, measure the prefix hit rate of a shared policy prompt, name the three cases where caching stops helping, and read the same counters from a running engine.

## Why it matters
- **Without it:** every support question re-sends the same 1,500-token policy, and the engine computes its KV again for each request. On the 30-request trace that is 45,000 prompt tokens of identical work, before any question is read.
- **With it:** the engine hashes each full block of the prompt. A block whose hash is already stored is reused, so only the question blocks are computed. In the simulation below, about 92% of the prompt tokens are reused.
- **Its limit:** only an exact token prefix matches. A different first word, a cache too small to hold the shared prefix, or no shared prefix at all gives no reuse.
- **When you don't need it:** every prompt is unique (no shared prefix), or the prompts are so short that prefill is cheap.

## The idea
Think of a train where each carriage is coupled to the one in front of it. You can reuse the carriages only from the front: if the first carriage is different, no carriage behind it can be reused, even if it looks the same. vLLM works this way with KV blocks. A block holds 16 tokens, and its hash covers those 16 tokens plus the hash of the block before it. Two prompts that share their first *k* full blocks share their first *k* hashes, so the engine reuses those KV blocks instead of computing them. A lookup starts at block 0 and stops at the first miss. Eviction is least recently used. SGLang uses the same exact-prefix rule with a radix tree (RadixAttention): a shared prefix is stored once as a path of nodes, and eviction removes unused leaves first.

## Picture
```mermaid
flowchart LR
  P["prompt: 1,500 policy tokens, then the question"] --> B["16-token blocks, full blocks only"]
  B --> H["hash chain: each hash covers its tokens and the previous hash"]
  H --> L{"stored? walk from block 0"}
  L -->|"hit: reuse the KV block"| R["no prefill for this block"]
  L -->|"miss: stop the walk"| C["compute the rest of the prompt"]
```

## Step by step
### Step 1: the trace and a stand-in policy
The trace is the made-up support-desk trace from lesson 07: 30 requests, each with a prompt of 1,500 policy tokens plus a question of 17 to 117 tokens. It was generated with a fixed seed and has no outside source, so no licence applies. The policy is a stand-in: the course has no copy of the real Brightside policy text, so its token ids are made up.

In [1]:
import json
from pathlib import Path

trace = [json.loads(line) for line in Path("../data/07/support_trace.jsonl").read_text().splitlines()]
POLICY_TOKENS = 1_500                                       # the shared prefix, the same in every request
print(f"{len(trace)} requests; first rows: {trace[:2]}")
print(f"question tokens: {min(r['prompt_tokens'] - POLICY_TOKENS for r in trace)} to "
      f"{max(r['prompt_tokens'] - POLICY_TOKENS for r in trace)}; all prompt tokens: {sum(r['prompt_tokens'] for r in trace):,}")

30 requests; first rows: [{'prompt_tokens': 1560, 'output_tokens': 24}, {'prompt_tokens': 1570, 'output_tokens': 45}]
question tokens: 17 to 117; all prompt tokens: 46,585


### Step 2: the engine's own hash chain (the library version)
The pinned vLLM CPU image (`vllm/vllm-openai-cpu:v0.32.0`, the same tag as lessons 08, 11 and 12) contains the hash function that its prefix cache uses. The probe below prints the defaults and checks the chain on two made-up prompts: 100 shared tokens, then a different tail. It needs no model.

In [2]:
LIBRARY_PROBE = """
from vllm.config import CacheConfig
from vllm.utils.hashing import get_hash_fn_by_name
from vllm.v1.core.kv_cache_utils import hash_block_tokens, init_none_hash
c = CacheConfig()
print('enable_prefix_caching default:', c.enable_prefix_caching, '| hash algo:', c.prefix_caching_hash_algo, '| block_size:', c.block_size)
fn = get_hash_fn_by_name(c.prefix_caching_hash_algo)
init_none_hash(fn)
def chain(tokens, size=16):
    parent, out = None, []
    for i in range(0, len(tokens) - len(tokens) % size, size):
        parent = hash_block_tokens(fn, parent, tokens[i:i + size])
        out.append(parent)
    return out
policy = list(range(1000, 1100))
a = chain(policy + [1, 2, 3])
b = chain(policy + [9, 9, 9])
print('full blocks:', len(a), len(b), '| equal leading hashes:', sum(x == y for x, y in zip(a, b)))
d = chain([7] + policy[1:] + [1, 2, 3])
print('first token changed, equal leading hashes:', sum(x == y for x, y in zip(a, d)))
print('hash bytes:', len(a[0]))
"""

**Run the probe in the pinned image.** Docker starts the image's Python with `-I`, so the run ignores the working folder. This cell is skipped unless Docker is enabled; when it runs, it prints its own output.

In [3]:
import subprocess

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"           # the pinned CPU image (lessons 08, 11 and 12 use the same tag)
run = subprocess.run(["docker", "run", "--rm", "--entrypoint", "python3", IMAGE, "-I", "-c", LIBRARY_PROBE],
                     capture_output=True, text=True, timeout=900)
print(run.stdout if run.returncode == 0 else "docker run failed: " + run.stderr[-300:])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**The recorded output.** Without Docker, this cell prints the recorded copy from a real run of the same image (`data/recorded/14.json`). It is a recording, not a live run.

In [4]:
import json
from pathlib import Path

recorded = json.loads(Path("../data/recorded/14.json").read_text())
print("recorded output (from a real run of " + recorded["image"] + "):")
print(recorded["stdout"])

recorded output (from a real run of vllm/vllm-openai-cpu:v0.32.0):
enable_prefix_caching default: True | hash algo: sha256 | block_size: 16
full blocks: 6 6 | equal leading hashes: 6
first token changed, equal leading hashes: 0
hash bytes: 32



**What you should see:** the default is `enable_prefix_caching: True` with the `sha256` hash and a block size of 16. The two prompts share their first six full blocks (96 tokens) and get six equal hashes. Changing the first token changes every hash, so no block matches. Each hash is 32 bytes.

## Under the hood
Step 2 used the engine's hash. Here is the same idea in about 35 lines: a hash chain over full blocks, and a cache that looks up leading blocks and evicts the least recently used one. It has no tenants, no eviction by time and no offload to CPU. Those come later in the course. The hashes below are a teaching version: they use sha256 over `repr(tokens)`, so their values do not match vLLM's hashes. The chain idea is the same.

First, the hash chain: one hash per full block, each covering the block's tokens and the hash before it.

In [5]:
import hashlib
from collections import OrderedDict

BLOCK = 16

def block_hashes(tokens, size=BLOCK):
    """One hash per full block. Each hash covers the block's tokens and the hash before it."""
    parent, out = b"", []
    for start in range(0, len(tokens) - len(tokens) % size, size):
        parent = hashlib.sha256(parent + repr(tokens[start:start + size]).encode()).digest()
        out.append(parent)
    return out

Next, the cache. A lookup walks the chain from block 0 and stops at the first miss. A store evicts the least recently used block when the cache is full.

In [6]:
class PrefixCache:
    """Full blocks by hash, with a capacity in blocks. A lookup walks from block 0 and stops at the first miss."""
    def __init__(self, capacity):
        self.capacity = capacity
        self.blocks = OrderedDict()                    # oldest first

    def lookup(self, hashes):
        hit = 0
        for h in hashes:
            if h not in self.blocks:
                break                                  # the chain is broken: nothing after this can match
            self.blocks.move_to_end(h)                 # used now: newest
            hit += 1
        return hit

    def store(self, hashes):
        for h in hashes:
            self.blocks.pop(h, None)                   # re-insert at the end: newest
            self.blocks[h] = True
            if len(self.blocks) > self.capacity:
                self.blocks.popitem(last=False)        # evict the least recently used block

### Step 3: run the trace through the cache
Each request builds its token ids: the same 1,500 policy ids, then its own question ids. Three variants change the picture: `question_first` puts the question in front, and `shared=False` gives every request its own policy. The counts are prompt tokens whose KV was reused.

In [7]:
import random

def make_prompt(i, row, question_first=False, shared=True):
    rng = random.Random(1000 + i)                      # seeded: the same question ids on every run
    question = [rng.randint(100_000, 199_999) for _ in range(row["prompt_tokens"] - POLICY_TOKENS)]
    base = 0 if shared else (i + 1) * 1_000_000        # unshared: a different policy per request
    policy = list(range(base + 1, base + POLICY_TOKENS + 1))
    return question + policy if question_first else policy + question

def run_trace(capacity=None, question_first=False, shared=True):
    cache = PrefixCache(capacity) if capacity else None
    hit_tokens = total = 0
    for i, row in enumerate(trace):
        hashes = block_hashes(make_prompt(i, row, question_first, shared))
        hit = cache.lookup(hashes) if cache else 0
        if cache:
            cache.store(hashes)
        hit_tokens += hit * BLOCK
        total += row["prompt_tokens"]
    return hit_tokens, total

Six setups on the same 30 requests. Each row runs the whole trace from an empty cache.

In [8]:
SETUPS = [("no cache", None, False, True), ("cache of 1,000 blocks", 1000, False, True),
          ("cache of 100 blocks", 100, False, True), ("cache of 60 blocks", 60, False, True),
          ("1,000 blocks, question first", 1000, True, True), ("1,000 blocks, no shared prefix", 1000, False, False)]
for name, cap, first, shared in SETUPS:
    hit, total = run_trace(cap, first, shared)
    print(f"{name}: {hit:,} of {total:,} prompt tokens reused ({hit / total:.1%})")

no cache: 0 of 46,585 prompt tokens reused (0.0%)
cache of 1,000 blocks: 43,152 of 46,585 prompt tokens reused (92.6%)
cache of 100 blocks: 41,664 of 46,585 prompt tokens reused (89.4%)
cache of 60 blocks: 0 of 46,585 prompt tokens reused (0.0%)
1,000 blocks, question first: 0 of 46,585 prompt tokens reused (0.0%)
1,000 blocks, no shared prefix: 0 of 46,585 prompt tokens reused (0.0%)


**What you should see:** the large cache reuses the 93 full policy blocks (1,488 tokens) in almost every request after the first: 92.6%. The 100-block cache gets 89.4%. The 60-block cache reuses nothing. The three cases where caching stops helping are all in the table: the first token differs (question first, 0%), the cache is smaller than the prefix (60 blocks, 0%), and there is no shared prefix at all (0%).

**Why 60 blocks gives 0%.** The walk starts at block 0. The policy is 93 blocks and the cache holds 60, so LRU evicts block 0 first. Every lookup misses at block 0 and stops.

**Why 100 blocks gives 89.4%, not 92.6%.** The newest request's blocks must fit beside the 93 policy blocks, and 100 blocks leave 7 spare. A question of 116 tokens or more fills 8 blocks. The cache then evicts policy block 0, and the request after it misses at block 0. That is the gap: one request loses its reuse, about 1,488 tokens.

### Step 4: the same counters from a running engine (optional)
A running vLLM server exposes two counters at `/metrics`: `vllm:prefix_cache_queries_total` (prompt tokens queried) and `vllm:prefix_cache_hits_total` (prompt tokens found in the cache). The hit rate of a set of calls is the difference in hits over the difference in queries. The cell sends two calls with the same stand-in policy to the engine in `VLLM_BASE_URL`, the engine from lesson 08. Without an engine it prints the error type and says so.

In [9]:
import sys, httpx; sys.path.insert(0, "..")
from openai import OpenAI
from tut_env import env

base = env("VLLM_BASE_URL", "http://localhost:8000/v1")   # lab value: the local engine from lesson 08
QUERIED, HITS = "vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total"

def counters():
    """Sum the two counters from /metrics. A line looks like name{labels} value; comments start with #."""
    vals = {QUERIED: 0.0, HITS: 0.0}
    for line in httpx.get(base.rsplit("/v1", 1)[0] + "/metrics", timeout=180).text.splitlines():
        name = line.split("{")[0].split(" ")[0]
        if name in vals:
            vals[name] += float(line.rsplit(" ", 1)[1])
    return vals

policy = "\n".join(f"Rule {i}: answer from the Brightside refund policy." for i in range(150))  # stand-in text
try:
    client = OpenAI(base_url=base, api_key="local", timeout=180, max_retries=0)  # the local engine does not check the key
    model, before = client.models.list().data[0].id, counters()
    for question in ("Can a customer get a refund after 30 days?", "Where is my refund?"):
        client.chat.completions.create(model=model, max_tokens=20, messages=[
            {"role": "system", "content": policy}, {"role": "user", "content": question}])
    after = counters()
    queried, hits = after[QUERIED] - before[QUERIED], after[HITS] - before[HITS]
    print(f"queried tokens: {queried:.0f} | cached tokens: {hits:.0f} | hit rate of these two calls: {hits / queried:.0%}")
except Exception as err:   # print the error type only, never the address
    print(f"engine unreachable or too slow ({type(err).__name__}): skipped. Start one in lesson 08.")

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Not run in the course's recording.** This step was not run against a generating engine when the course was recorded, so no output is shown here. If you start an engine from lesson 08, the second call should reuse the policy blocks and the rate should be above zero. The exact value depends on the block size and the prompt length; the startup log shows the block size. The `/metrics` page has lines of this shape (`<n>` stands for a number; the label set varies by version):

```
vllm:prefix_cache_queries_total{...} <n>
vllm:prefix_cache_hits_total{...} <n>
```

For an A/B test, start the same server with `--no-enable-prefix-caching`.

## At work
- **Cloudflare** reports that an `x-session-affinity` header raised the peak input cache hit ratio from 60% to 80%. Cached blocks live on one engine, so the router must send a session back to the same engine. → lesson 30 (routing).
- **Meta and PyTorch** report a 40 to 50% prefix cache hit rate from sticky routing at 90% HBM utilisation. → lesson 34 (disaggregated serving).
- **Shared caches are a side channel.** A timing difference on a hit can reveal another tenant's prompt. `cache_salt` keeps blocks apart per tenant. → lesson 36 (tenancy).

Sources: Cloudflare, "High-performance LLMs" (2026-05), https://blog.cloudflare.com/high-performance-llms/; PyTorch blog (2025-09-12), https://pytorch.org/blog/disaggregated-inference-at-scale-with-pytorch-vllm/; vLLM design doc, https://docs.vllm.ai/en/latest/design/prefix_caching/.

## Common mistakes
**1. Putting the question before the policy.** The first block then differs for every request, and nothing after it can match. Assert the hit rate and see the failure.

In [10]:
hit_tokens, total = run_trace(capacity=1000, question_first=True)
assert hit_tokens / total > 0.5, f"hit rate {hit_tokens / total:.0%}: the shared text must come first"
# AssertionError: hit rate 0%: the shared text must come first

AssertionError: hit rate 0%: the shared text must come first

**2. Sizing the cache below the shared prefix.** LRU keeps recent blocks, so a cache smaller than the 93 policy blocks evicts the start of the prefix before the next request arrives.

In [11]:
hit_tokens, total = run_trace(capacity=60)
assert hit_tokens / total > 0.5, f"hit rate {hit_tokens / total:.0%}: the cache holds fewer blocks than the shared prefix"
# AssertionError: hit rate 0%: the cache holds fewer blocks than the shared prefix

AssertionError: hit rate 0%: the cache holds fewer blocks than the shared prefix

## Try it
Predict first: what is the smallest cache, in blocks, that keeps the hit rate above 80%? Try the values 60, 80, 93, 95, 97, 98, 100 and 120, then run the cell and compare.

In [12]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [13]:
for capacity in (60, 80, 93, 95, 97, 98, 100, 120):
    hit, total = run_trace(capacity)
    print(f"cache of {capacity:>3} blocks: hit rate {hit / total:.1%}")
# Answer: 98 blocks. The policy is 93 blocks, so 98 leaves 5 spare for a request's new blocks.
# At 93 the first policy block is evicted by the request's own question blocks, and the rate is zero.
# Between 94 and 98 some requests still hit: a request that misses stores the whole policy again,
# and the next request can then hit. The rate reaches 92.6% at 101 blocks, which holds the policy
# plus the longest question (8 blocks).

cache of  60 blocks: hit rate 0.0%
cache of  80 blocks: hit rate 0.0%
cache of  93 blocks: hit rate 0.0%
cache of  95 blocks: hit rate 19.2%


cache of  97 blocks: hit rate 73.5%
cache of  98 blocks: hit rate 86.2%
cache of 100 blocks: hit rate 89.4%
cache of 120 blocks: hit rate 92.6%


## Recap
- A block's hash chains over everything before it, so a block matches only when its whole prefix matches.
- A lookup starts at block 0 and stops at the first miss. Three cases stop the reuse: the first token differs (put static text first), the cache is smaller than the shared prefix (LRU evicts block 0), or no two prompts share a prefix.
- LRU keeps the prefix hot only if the cache holds it plus the newest request's blocks.
- Hit rate = change in `prefix_cache_hits_total` over change in `prefix_cache_queries_total`.

**Interview:** "How does automatic prefix caching decide that two requests share a prefix, and why does a change in the first token remove every hit?" Strong answer: each block's hash covers its tokens and the previous hash; a lookup walks from block 0 and stops at the first miss, so a changed early token changes every later hash. Put shared content first.